# 03 — Build dimensions
Builds `silver.airports` (334 rows) and `silver.airlines` (15 rows) —
the things that *exist*, as opposed to flights, which *happen*.

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("flights.bronze.flights_raw")

# Airports — from both origin and destination columns
origins = df.select(
    F.col("Origin").alias("airport_code"),
    F.col("OriginCityName").alias("city_name"),
    F.col("OriginState").alias("state_code"),
    F.col("OriginStateName").alias("state_name")
).distinct()

dests = df.select(
    F.col("Dest").alias("airport_code"),
    F.col("DestCityName").alias("city_name"),
    F.col("DestState").alias("state_code"),
    F.col("DestStateName").alias("state_name")
).distinct()

airports = origins.union(dests).distinct()

airports.write.mode("overwrite").saveAsTable("flights.silver.airports")

# Airlines — codes from the data, names typed by hand
airlines = df.select(
    F.col("Reporting_Airline").alias("airline_code"),
    F.col("DOT_ID_Reporting_Airline").alias("dot_id")
).distinct()

airline_names = spark.createDataFrame([
    ("WN", "Southwest Airlines"),
    ("AA", "American Airlines"),
    ("DL", "Delta Air Lines"),
    ("UA", "United Airlines"),
    ("OO", "SkyWest Airlines"),
    ("YX", "Republic Airways"),
    ("MQ", "Envoy Air"),
    ("NK", "Spirit Airlines"),
    ("B6", "JetBlue Airways"),
    ("AS", "Alaska Airlines"),
    ("9E", "Endeavor Air"),
    ("F9", "Frontier Airlines"),
    ("G4", "Allegiant Air"),
    ("OH", "PSA Airlines"),
    ("HA", "Hawaiian Airlines"),
], ["airline_code", "airline_name"])

airlines_full = airlines.join(airline_names, "airline_code", "left")

airlines_full.write.mode("overwrite").saveAsTable("flights.silver.airlines")

In [0]:
spark.read.table("flights.silver.airlines").filter(F.col("airline_name").isNull()).show()